# Inferencia individual: scoring de un crédito con escalado a agente

Este notebook simula el **servicio de scoring** que correría en producción. Carga los tres artefactos serializados por el notebook `12_complete_pipeline.ipynb` (`base_pre.pkl`, `base_filter.pkl`, `va_cal.pkl`), toma uno o varios créditos en bruto del CSV de test y los pasa por el pipeline completo:

```
CSV crudo  ──▶  base_pre.transform  ──▶  base_filter.transform  ──▶  va_cal.predict_proba
```

Para cada crédito devolvemos:

- **`p_0`** — límite inferior del par de Venn-Abers (probabilidad bajo la hipótesis $y=0$).
- **`p_1`** — límite superior del par de Venn-Abers (probabilidad bajo la hipótesis $y=1$).
- **`p`** — probabilidad puntual calibrada (media de `p_0` y `p_1`).
- **`decision`** — política de negocio basada en la **anchura** del intervalo Venn-Abers:
  - si `p_1 - p_0 > 0.2` → **`escalar_a_agente`** (el modelo no está seguro: revisión manual).
  - en caso contrario, decisión automática con `p`:
    - `p ≥ 0.5` → **`default`**
    - `p < 0.5` → **`no_default`**

Idea clave: Venn-Abers garantiza que la probabilidad real cae dentro de `[p_0, p_1]`. Si ese intervalo es ancho (> 0.2 = 20 puntos porcentuales), el modelo admite que **no sabe lo suficiente** sobre ese crédito y la decisión se delega a un analista humano. Si el intervalo es estrecho, el modelo está seguro y se decide automáticamente con la probabilidad puntual.

## 1 — Carga de los artefactos serializados

Cargamos los tres `pkl` generados por el notebook 12. Importamos antes las clases originales (`BasePreprocess`, `BaseFiltering`, `VennAbersCalibrator`) porque `pickle` necesita reconstruirlas para deserializar correctamente.

Tras este bloque tenemos:
- `base_pre`: preprocesador con todas las transformaciones aprendidas en train.
- `base_filter`: filtro con la lista exacta de columnas a conservar.
- `va_cal`: calibrador Venn-Abers que envuelve internamente al `CatBoostClassifier` óptimo.

In [1]:
import pickle
import numpy as np
import pandas as pd

from src.preprocessing.base_preprocessing import BasePreprocess
from src.filtering.base_filtering import BaseFiltering
from venn_abers import VennAbersCalibrator

with open("preprocessor.pkl", "rb") as f:
    base_pre = pickle.load(f)

with open("filter.pkl", "rb") as f:
    base_filter = pickle.load(f)

with open("va_cal.pkl", "rb") as f:
    va_cal = pickle.load(f)

print("Preprocesador :", type(base_pre).__name__)
print("Filtro        :", type(base_filter).__name__,
      "| n_features_final:", base_filter.n_features_final)
print("Calibrador    :", type(va_cal).__name__,
      "| modelo base:", type(va_cal.estimator).__name__)

Preprocesador : BasePreprocess
Filtro        : BaseFiltering | n_features_final: 203
Calibrador    : VennAbersCalibrator | modelo base: CatBoostClassifier


## 2 — Lectura de los datos en bruto de test

Leemos `data/df_test_small.csv`, exactamente el mismo formato crudo que vio el preprocesador en `fit`. Para esta demo seleccionamos las **5 primeras filas** como si fueran 5 nuevas solicitudes de crédito que llegan al servicio.

> Las pasamos al pipeline directamente como `DataFrame` (sin pasar por disco).

In [ ]:
df_test = pd.read_csv("data/df_test_small.csv")
print(f"Test completo: {df_test.shape[0]} filas x {df_test.shape[1]} columnas")

# Tomamos 5 creditos como ejemplo de inferencia. Filtramos por `desc` no nulo
# para que la muestra sea representativa de los datos que vio el modelo en
# train (la mayoria de creditos con descripcion textual).
sample = df_test[df_test['desc'].notna()].head(5).copy()
print(f"\nMuestra de inferencia: {sample.shape[0]} filas x {sample.shape[1]} columnas")

## 3 — Función de scoring end-to-end

Encapsulamos todo el pipeline en una sola función `score_creditos(csv_path)`. Esta es la pieza que en producción expondría un endpoint HTTP: recibe un CSV, ejecuta `transform → transform → predict_proba` y devuelve la decisión por crédito.

**Política de decisión** (basada en la anchura del intervalo Venn-Abers, no en la probabilidad puntual):

| Condición                        | Decisión             |
|----------------------------------|----------------------|
| `p_1 - p_0 > 0.2`                | `escalar_a_agente`   |
| intervalo estrecho y `p ≥ 0.5`   | `default`            |
| intervalo estrecho y `p < 0.5`   | `no_default`         |

Solo se decide automáticamente cuando el modelo está **seguro** (intervalo Venn-Abers estrecho). En cuanto la incertidumbre es grande (anchura > 20 pp), el caso se delega al analista humano, que recibe `p_0`, `p_1` y `p` para tomar la decisión final.

In [ ]:
UMBRAL_DEFAULT = 0.5     # p >= 0.5 -> default (cuando hay certeza)
UMBRAL_ANCHURA = 0.2     # p_1 - p_0 > 0.2 -> escalar a agente


def decidir(p: float, p_0: float, p_1: float) -> str:
    # Si Venn-Abers no esta seguro (intervalo ancho), delegamos en el analista.
    if (p_1 - p_0) > UMBRAL_ANCHURA:
        return "escalar_a_agente"
    # Intervalo estrecho: el modelo esta seguro, decidimos con la prob puntual.
    if p >= UMBRAL_DEFAULT:
        return "default"
    return "no_default"


def score_creditos(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica preproc -> filtro -> modelo calibrado a un DataFrame en bruto.

    `df` debe tener las mismas columnas que el set de entrenamiento (la columna
    `loan_status` es opcional en inferencia). Devuelve un DataFrame con
    p_0, p, p_1, decision por crédito.
    """
    # 1) Preprocesamiento (mismo fit aprendido en train).
    X_pre, _ = base_pre.transform(df)

    # 2) Filtrado (mismas columnas seleccionadas en train).
    X_filt = base_filter.transform(X_pre)

    # 3) Probabilidad puntual calibrada + par Venn-Abers (p_0, p_1).
    proba = va_cal.predict_proba(X_filt)
    p = proba[:, 1]

    _, p0p1 = va_cal.predict_proba(X_filt, p0_p1_output=True)
    p0p1 = np.asarray(p0p1)
    p_0 = p0p1[:, :, 0].mean(axis=0)
    p_1 = p0p1[:, :, 1].mean(axis=0)

    decisiones = [decidir(pi, p0i, p1i) for pi, p0i, p1i in zip(p, p_0, p_1)]

    return pd.DataFrame({
        "p_0":      p_0,
        "p":        p,
        "p_1":      p_1,
        "decision": decisiones,
    })

## 4 — Aplicación a la muestra

Lanzamos la función sobre los 5 créditos de prueba. La salida es exactamente lo que devolvería el servicio en producción para una petición de scoring.

In [ ]:
resultado = score_creditos(sample)
resultado

## 5 — Vista detallada por crédito

Imprimimos el resultado de cada crédito en formato de informe legible, tal como aparecería en la UI de un analista de riesgos.

In [ ]:
for i, row in resultado.iterrows():
    anchura = row['p_1'] - row['p_0']
    print(f"Credito #{i}")
    print(f"  p_0      : {row['p_0']:.4f}")
    print(f"  p        : {row['p']:.4f}")
    print(f"  p_1      : {row['p_1']:.4f}")
    print(f"  anchura  : {anchura:.4f}")
    print(f"  decision : {row['decision']}")
    print("-" * 40)